### 作業フォルダの指定

In [ ]:
from pathlib import Path
import os

build_dir = Path("../build").resolve()
build_dir.mkdir(exist_ok=True)

os.chdir(build_dir)

### OpenModelica接続確認

In [ ]:
from OMPython import OMCSession

omc = OMCSession.OMCSessionLocal()

print(omc.sendExpression("getVersion()"))

### モデルファイルの場所の確認

In [ ]:
from pathlib import Path

model_file = Path("../models/MassSpringDamper.mo").resolve()

# print(model_file)
print(model_file.exists())

### モデルの読み込み

In [ ]:
omc.sendExpression(
    f'loadFile("{model_file}")'
)

### モデルのチェック

In [ ]:
omc.sendExpression(
    "checkModel(MassSpringDamper)"
)

### モデルの中身確認

In [ ]:
omc.sendExpression(
    "getClassNames()"
)

### シミュレーション

In [ ]:
result = omc.sendExpression("""
simulate(
    MassSpringDamper,
    startTime=0,
    stopTime=10,
    numberOfIntervals=1000
)
""")

result

### 結果ファイル取得

In [ ]:
result_file = Path(result["resultFile"]).resolve()

# print(result_file)
print(result_file.exists())

### MAT ファイル読み込み

In [ ]:
from DyMat import DyMatFile

data = DyMatFile(result_file)

### 利用可能な変数確認

In [ ]:
for name in sorted(data.names()):
    print(name)

### 変位データ取得

In [ ]:
x = data.data("Mass.s")
t = data.abscissa("Mass.s")[0]

### グラフ描画

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.plot(t, x)

plt.title("Mass-Spring-Damper")
plt.xlabel("Time [s]")
plt.ylabel("Displacement [m]")

plt.grid(True)

plt.show()

### 速度も表示

In [ ]:
v = data.data("Mass.v")

plt.figure(figsize=(10,4))

plt.plot(t, v)

plt.title("Velocity")
plt.xlabel("Time [s]")
plt.ylabel("Velocity [m/s]")

plt.grid(True)

plt.show()

### pandas に変換

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "time": t,
    "displacement": x,
    "velocity": v
})

df.head()

### CSV 出力

In [ ]:
# 結果ディレクトリを作成
from pathlib import Path

result_dir = Path("../results")

result_dir.mkdir(exist_ok=True)

# 出力
csv_file = result_dir / "mass_spring_damper.csv"

df.to_csv(
    csv_file,
    index=False
)

print(csv_file)

### CSV 内容確認

In [ ]:
pd.read_csv(csv_file).head()

### 最大変位を計算

In [ ]:
max_disp = abs(x).max()

print(max_disp)

### サマリー CSV

In [ ]:
summary = pd.DataFrame([
    {
        "k": 1000,
        "c": 20,
        "m": 10,
        "max_displacement": abs(x).max(),
        "final_displacement": x[-1]
    }
])

summary

# 保存
summary.to_csv(
    result_dir / "summary.csv",
    index=False
)